# 05 — Vintage tracking (downturn vs calm)

**Plain English:** The strongest story in this data: line the vintages up by
*months on book* (not calendar time) and watch how fast each cohort goes bad.
The 2007 and 2008 cohorts originated straight into the financial crisis; 2015
originated into a calm market. The cumulative default curves separate hard — the
clearest demonstration of why vintage tracking matters.

**One-line terms**
- **Months on book** — loan age in months since first payment; puts every vintage on a common clock.
- **Cumulative default rate** — share of the cohort that has reached Stage 3 (90+/credit event) by a given age.
- **Vintage** — origination-year cohort.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# First month each loan reaches default (Stage 3), by loan age ----------
defaulted = panel[panel.stage == "Stage 3"]
first_def = defaulted.groupby(["vintage", "loan_seq"])["loan_age"].min().reset_index()
loans_per_vintage = panel.groupby("vintage")["loan_seq"].nunique()

ages = range(0, 121)  # 0..120 months on book
curves = {}
for v in m.VINTAGES:
    fv = first_def[first_def.vintage == v]
    n = loans_per_vintage[v]
    cum = [(fv.loan_age <= a).sum() / n for a in ages]
    curves[v] = cum
curve_df = pd.DataFrame(curves, index=list(ages))
curve_df.index.name = "months_on_book"
curve_df.round(4).head(13)

,2007,2008,2015
months_on_book,,,
0,0.0000,0.0001,0.0000
1,0.0006,0.0002,0.0000
2,0.0010,0.0004,0.0001
3,0.0037,0.0020,0.0005
4,0.0064,0.0039,0.0006
5,0.0087,0.0054,0.0011
6,0.0113,0.0072,0.0014
7,0.0135,0.0088,0.0017
8,0.0153,0.0107,0.0018


In [3]:
# Plot the cumulative default curves ------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for v in m.VINTAGES:
    ax.plot(curve_df.index, 100 * curve_df[v], label=f"{v} vintage", linewidth=2)
ax.set_xlabel("months on book"); ax.set_ylabel("cumulative % ever 90+/default")
ax.set_title("Cumulative default by months on book — downturn (2007/08) vs calm (2015)")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(m.OUT_CHARTS / "05_vintage_default_curves.png", dpi=130)
print("saved curve chart"); plt.close(fig)

saved curve chart


In [4]:
# Clean table: cumulative default at selected ages ----------------------
marks = [12, 24, 36, 48, 60, 72]
snap = (curve_df.loc[curve_df.index.isin(marks)]
        .mul(100).round(2).reset_index())
snap.columns = ["months_on_book"] + [f"{v}_cum_default_pct" for v in m.VINTAGES]
snap.to_csv(m.OUT_TABLES / "05_vintage_cumulative_default.csv", index=False)
snap

,months_on_book,2007_cum_default_pct,2008_cum_default_pct,2015_cum_default_pct
0,12,2.45,1.89,0.30
1,24,6.18,4.47,0.68
2,36,9.86,6.03,1.02
3,48,12.20,7.19,1.23
4,60,13.74,7.84,2.70
5,72,14.64,8.20,3.65
